[Énoncés : Série 4 — Piles et files](../serie4_piles_files.ipynb)

# Corrigé — Série 4 — Piles et files

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 19 — Simplifier un chemin de dossiers

**Idée** : découper sur `/` et empiler les noms de dossiers. `..` dépile (si la pile n'est pas vide), `.` et les morceaux vides sont ignorés. À la fin, la pile contient le chemin, du dossier racine au dossier final.

**Complexité** : O(n).

In [ ]:
def simplifier_chemin(chemin: str) -> str:
    pile = []
    for morceau in chemin.split("/"):      # "//" et le "/" final donnent des morceaux vides
        if morceau == "" or morceau == ".":
            continue
        if morceau == "..":
            if pile:                        # à la racine, ".." ne fait rien
                pile.pop()
        else:
            pile.append(morceau)
    return "/" + "/".join(pile)


verifier(simplifier_chemin, [
    (('/a/b/../c/./',), '/a/c'),
    (('/',), '/'),
    (('/../',), '/'),   # on ne remonte pas au-dessus de la racine
    (('//home//user/',), '/home/user'),
    (('/a/./b/../../c/',), '/c'),
    (('/.../a',), '/.../a'),   # ... est un nom de dossier
    (('/a/b/c/../../..',), '/'),
])

## Exercice 20 — Corriger les frappes

**Idée** : une pile de caractères. Un caractère normal est empilé, un `#` dépile (si possible). C'est le même schéma que `supprimer_paires` du thème 4.

**Complexité** : O(n).

In [ ]:
def appliquer_retouches(texte: str) -> str:
    pile = []
    for c in texte:
        if c == "#":
            if pile:
                pile.pop()          # efface le dernier caractère encore présent
        else:
            pile.append(c)
    return "".join(pile)


verifier(appliquer_retouches, [
    (('ab#c',), 'ac'),
    (('a##b',), 'b'),
    (('',), ''),
    (('###',), ''),   # rien à effacer
    (('abc',), 'abc'),
    (('abc###d#e',), 'e'),
    (('x#y#z',), 'z'),
])

## Exercice 21 — Calculatrice postfixée

**Idée** : une pile de nombres. Un nombre est empilé ; un opérateur dépile ses deux opérandes, calcule, et rempile le résultat. À la fin, il reste un seul nombre : le résultat. Piège : le premier élément dépilé est l'opérande de **droite**.

**Complexité** : O(n).

In [ ]:
def evaluer_postfixe(tokens: list[str]) -> int:
    pile = []
    for t in tokens:
        if t in ("+", "-", "*"):
            droite = pile.pop()        # attention : le PREMIER dépilé est l'opérande de droite
            gauche = pile.pop()
            if t == "+":
                pile.append(gauche + droite)
            elif t == "-":
                pile.append(gauche - droite)
            else:
                pile.append(gauche * droite)
        else:
            pile.append(int(t))        # un nombre : on le garde en attendant son opérateur
    return pile[0]


verifier(evaluer_postfixe, [
    ((['2', '3', '+', '4', '*'],), 20),
    ((['5', '2', '-'],), 3),   # 5 - 2, pas 2 - 5
    ((['7'],), 7),
    ((['2', '3', '4', '*', '+'],), 14),
    ((['10', '2', '3', '*', '-'],), 4),
    ((['1', '5', '-', '3', '*'],), -12),
    ((['4', '13', '5', '*', '+'],), 69),
])

## Exercice 22 — Ordre de fin des tâches

**Idée** : simuler la file avec un `deque` (thème 5) contenant des paires `(indice, travail restant)`. `popleft()` prend la tâche en tête, `append()` la remet à la fin.

**Complexité** : O(total des durées / quantum + n). Avec une liste et `pop(0)`, chaque retrait coûterait O(n).

In [ ]:
from collections import deque

def ordre_fin_taches(durees: list[int], quantum: int) -> list[int]:
    file = deque(enumerate(durees))      # des paires (indice, travail restant)
    ordre = []
    while file:
        i, reste = file.popleft()        # la tâche en tête de file
        reste -= quantum
        if reste <= 0:
            ordre.append(i)              # terminée
        else:
            file.append((i, reste))      # pas finie : retour en fin de file
    return ordre


verifier(ordre_fin_taches, [
    (([3, 1, 2], 2), [1, 2, 0]),
    (([1, 1, 1], 5), [0, 1, 2]),
    (([5], 1), [0]),
    (([], 3), []),
    (([4, 2, 6], 2), [1, 0, 2]),
    (([3, 3, 3], 1), [0, 1, 2]),
    (([2, 5, 1, 3], 2), [0, 2, 3, 1]),
])

## Exercice 23 — Profondeur des parenthèses

**Idée** : la profondeur, c'est la hauteur de la pile des parenthèses ouvertes. Comme on n'a pas besoin de savoir **quelles** parenthèses sont dans la pile, un simple compteur suffit : +1 à l'ouverture, −1 à la fermeture, et on retient le maximum atteint.

**Complexité** : O(n).

In [ ]:
def profondeur_parentheses(texte: str) -> int:
    profondeur = 0           # nombre de parenthèses ouvertes non encore fermées
    maximum = 0
    for c in texte:
        if c == "(":
            profondeur += 1
            maximum = max(maximum, profondeur)
        elif c == ")":
            profondeur -= 1
    return maximum


verifier(profondeur_parentheses, [
    (('a(b(c)d)e',), 2),
    (('()',), 1),
    (('abc',), 0),   # aucune parenthèse
    (('',), 0),
    (('(1)+((2))+(((3)))',), 3),
    (('()()()',), 1),
])

---

[Énoncés de la série](../serie4_piles_files.ipynb)